# KKBOX Tab 3 Monte Carlo Scenario Catalog

- Notebook tự chứa để build nhiều preset Monte Carlo artifact cho Tab 3.
- Đầu vào chính là feature store canonical và artifact Tab 2 đã score.
- Monte Carlo ở đây là lớp `proxy uncertainty` cho doanh thu và risk sau can thiệp, không phải causal uplift model.
- Notebook không phụ thuộc `.py` helper bên ngoài khi chạy trên Kaggle.
- Output dùng cho web demo artifact-backed, chỉ cần load theo `scenario_id`.


In [ ]:
from __future__ import annotations

import json
from copy import deepcopy
from datetime import UTC, datetime
from pathlib import Path
from typing import Any, Iterable, Sequence

import numpy as np
import pandas as pd
from IPython.display import display


In [ ]:
FEATURE_STORE_ROOT_HINT = None
TAB2_ARTIFACTS_ROOT_HINT = None
OUTPUT_DIR = 'artifacts_tab3_monte_carlo'
SCORE_MONTH = 201704

SCENARIO_CONFIG = {
    'manual_to_auto_share': 0.30,
    'upsell_share': 0.20,
    'engagement_share': 0.25,
    'manual_to_auto_cost_per_user': 0.0,
    'upsell_cost_per_user': 0.0,
    'engagement_cost_per_user': 0.0,
}

MC_CONFIG = {
    'n_iterations': 1000,
    'seed': 42,
    'beta_concentration': 250.0,
    'auto_effect_std': 0.15,
    'engagement_effect_std': 0.20,
    'upsell_risk_std': 0.20,
    'upsell_amount_std': 0.10,
}

SIMULATION_UNIT_KEYS = [
    'bi_segment_name',
    'risk_band',
    'renewal_segment',
    'price_segment',
    'eligible_auto',
    'eligible_upsell',
    'eligible_engagement',
]


In [ ]:
def discover_project_dir(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / 'project-realtime-bi').exists():
            return (candidate / 'project-realtime-bi').resolve()
        if (candidate / 'data').exists() and (candidate / 'notebooks').exists():
            return candidate.resolve()
    return start


def build_upward_candidates(base_dir: Path, relative_parts: Sequence[str]) -> list[Path]:
    return [parent.joinpath(*relative_parts) for parent in [base_dir.resolve(), *base_dir.resolve().parents]]


def resolve_feature_store_dir(root_hint: str | Path | None = None, score_month: int = 201704) -> Path:
    required = [
        'train_features_bi_all.parquet',
        f'test_features_bi_{score_month}_full.parquet',
        'feature_columns.csv',
        'bi_dimension_columns.csv',
    ]
    base_dir = discover_project_dir()
    candidates: list[Path] = []
    if root_hint is not None:
        root_hint = Path(root_hint)
        candidates.extend([root_hint, root_hint / 'feature_store'])
    candidates.extend([
        Path('/kaggle/input/kkbox-feature-store'),
        Path('/kaggle/input/kkbox-feature-store/feature_store'),
        Path('/kaggle/input/kkbox-churn-feature-store'),
        Path('/kaggle/input/kkbox-churn-feature-store/feature_store'),
        Path('/kaggle/input/kkbox-churn-output'),
        Path('/kaggle/input/kkbox-churn-output/feature_store'),
    ])
    candidates.extend(build_upward_candidates(base_dir, ('data', 'artifacts', 'feature_store')))
    candidates.extend(build_upward_candidates(base_dir, ('artifacts', 'feature_store')))
    candidates.extend(build_upward_candidates(base_dir, ('feature_store',)))
    seen: set[Path] = set()
    for candidate in candidates:
        candidate = candidate.resolve()
        if candidate in seen or not candidate.exists():
            continue
        seen.add(candidate)
        if all((candidate / name).exists() for name in required):
            return candidate
    raise FileNotFoundError(f'Khong tim thay feature store canonical can cac file: {required}')


def resolve_tab2_artifacts_dir(root_hint: str | Path | None = None, score_month: int = 201704) -> Path:
    required = [
        f'tab2_test_scored_{score_month}.parquet',
        'tab2_validation_metrics.json',
        'tab2_model_summary.json',
    ]
    base_dir = discover_project_dir()
    candidates: list[Path] = []
    if root_hint is not None:
        root_hint = Path(root_hint)
        candidates.extend([root_hint, root_hint / 'artifacts_tab2_predictive'])
    candidates.extend([
        Path('/kaggle/input/kkbox-tab2-predictive'),
        Path('/kaggle/input/kkbox-tab2-predictive/artifacts_tab2_predictive'),
        Path('/kaggle/input/kkbox-tab2-predictive-artifacts'),
        Path('/kaggle/input/kkbox-tab2-predictive-artifacts/artifacts_tab2_predictive'),
    ])
    candidates.extend(build_upward_candidates(base_dir, ('data', 'artifacts_tab2_predictive')))
    candidates.extend(build_upward_candidates(base_dir, ('data', 'artifacts', 'tab2_predictive')))
    candidates.extend(build_upward_candidates(base_dir, ('artifacts_tab2_predictive',)))
    seen: set[Path] = set()
    for candidate in candidates:
        candidate = candidate.resolve()
        if candidate in seen or not candidate.exists():
            continue
        seen.add(candidate)
        if all((candidate / name).exists() for name in required):
            return candidate
    raise FileNotFoundError(f'Khong tim thay Tab 2 predictive artifacts can cac file: {required}')


def ensure_output_dir(output_dir: str | Path) -> Path:
    path = Path(output_dir).resolve()
    path.mkdir(parents=True, exist_ok=True)
    return path


def to_json_serializable(value: Any):
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return float(value)
    if isinstance(value, (np.bool_,)):
        return bool(value)
    if isinstance(value, datetime):
        return value.isoformat()
    if isinstance(value, pd.Timestamp):
        return value.isoformat()
    if isinstance(value, dict):
        return {str(key): to_json_serializable(val) for key, val in value.items()}
    if isinstance(value, (list, tuple)):
        return [to_json_serializable(item) for item in value]
    return value


def write_json(path: str | Path, payload: dict) -> Path:
    path = Path(path).resolve()
    path.write_text(json.dumps(to_json_serializable(payload), indent=2, ensure_ascii=False), encoding='utf-8')
    return path


def write_manifest(output_dir: str | Path, notebook_name: str, artifact_type: str, input_paths: dict[str, str | Path], output_paths: dict[str, str | Path], metadata: dict | None = None) -> Path:
    output_dir = ensure_output_dir(output_dir)
    payload = {
        'notebook_name': notebook_name,
        'artifact_type': artifact_type,
        'created_at_utc': datetime.now(UTC).isoformat(),
        'inputs': {key: str(value) for key, value in input_paths.items()},
        'outputs': {key: str(value) for key, value in output_paths.items()},
        'metadata': metadata or {},
    }
    return write_json(output_dir / 'manifest.json', payload)


def risk_band_from_probability(probabilities: pd.Series) -> pd.Series:
    probs = probabilities.astype('float32').clip(lower=0.0, upper=1.0)
    return pd.Series(
        np.select(
            [
                probs >= 0.80,
                probs >= 0.60,
                probs >= 0.40,
                probs >= 0.20,
            ],
            ['Very High', 'High', 'Medium', 'Low'],
            default='Very Low',
        ),
        index=probabilities.index,
    )


In [ ]:
DEFAULT_SCENARIO_CONFIG = deepcopy(SCENARIO_CONFIG)
DEFAULT_SENSITIVITY_SHARES = [0.05, 0.10, 0.20, 0.30, 0.40, 0.50]


def ensure_columns(df: pd.DataFrame, required: Iterable[str], label: str) -> None:
    missing = [col for col in required if col not in df.columns]
    if missing:
        raise ValueError(f'{label} thiếu cột bắt buộc: {missing}')


def _clip_share(value: float | int | None) -> float:
    if value is None:
        return 0.0
    return float(np.clip(float(value), 0.0, 1.0))


def _safe_probability(series: pd.Series) -> pd.Series:
    return series.fillna(0).astype('float32').clip(lower=1e-4, upper=1 - 1e-4)


def _safe_amount(series: pd.Series) -> pd.Series:
    return series.fillna(0).astype('float32').clip(lower=0.0)


def _mean_or_default(series: pd.Series, default: float) -> float:
    valid = series.dropna()
    if valid.empty:
        return float(default)
    return float(valid.mean())


def _first_non_empty_median(frames: list[pd.Series], default: float = 0.0) -> float:
    for series in frames:
        valid = series.dropna()
        if not valid.empty:
            return float(valid.median())
    return float(default)


def load_feature_snapshot(feature_store_dir: str | Path, score_month: int) -> pd.DataFrame:
    feature_store_dir = Path(feature_store_dir)
    score_path = feature_store_dir / f'test_features_bi_{score_month}_full.parquet'
    if score_path.exists():
        df = pd.read_parquet(score_path)
        if 'target_month' in df.columns:
            df = df[df['target_month'] == score_month].reset_index(drop=True)
        return df
    master_path = feature_store_dir / 'bi_feature_master.parquet'
    if master_path.exists():
        df = pd.read_parquet(master_path)
        return df[df['target_month'] == score_month].reset_index(drop=True)
    raise FileNotFoundError(f'Không tìm thấy snapshot BI cho score_month={score_month}.')


def _prepare_baseline_dataframe(feature_df: pd.DataFrame, scored_df: pd.DataFrame) -> pd.DataFrame:
    ensure_columns(scored_df, ('msno', 'target_month', 'churn_probability', 'expected_renewal_amount'), 'Tab 2 scored artifact')
    ensure_columns(feature_df, ('msno', 'target_month'), 'feature snapshot')
    feature_columns = [
        'msno', 'target_month', 'renewal_segment', 'price_segment', 'loyalty_segment', 'active_segment',
        'skip_segment', 'discovery_segment', 'rfm_segment', 'bi_segment_name', 'is_auto_renew', 'is_manual_renew',
        'deal_hunter_flag', 'free_trial_flag', 'high_skip_flag', 'low_discovery_flag', 'content_fatigue_flag',
        'skip_ratio', 'discovery_ratio', 'expected_renewal_amount', 'amt_per_day', 'rfm_total_score',
    ]
    feature_only_cols = [col for col in feature_columns if col in feature_df.columns and col not in scored_df.columns]
    merged = scored_df.merge(feature_df[['msno', 'target_month', *feature_only_cols]], on=['msno', 'target_month'], how='left')
    merged['baseline_churn_probability'] = _safe_probability(merged['churn_probability'])
    merged['expected_renewal_amount'] = _safe_amount(merged['expected_renewal_amount'])

    if 'risk_band' not in merged.columns:
        merged['risk_band'] = risk_band_from_probability(merged['baseline_churn_probability'])
    else:
        merged['risk_band'] = merged['risk_band'].fillna('Unknown').astype(str)

    for col in ('renewal_segment', 'price_segment', 'loyalty_segment', 'active_segment', 'skip_segment', 'discovery_segment', 'rfm_segment', 'bi_segment_name'):
        if col not in merged.columns:
            merged[col] = 'Unknown'
        merged[col] = merged[col].fillna('Unknown').astype(str)

    if 'skip_ratio' not in merged.columns:
        merged['skip_ratio'] = 0.0
    if 'discovery_ratio' not in merged.columns:
        merged['discovery_ratio'] = 0.0
    merged['skip_ratio'] = merged['skip_ratio'].fillna(0).astype('float32').clip(lower=0.0, upper=1.0)
    merged['discovery_ratio'] = merged['discovery_ratio'].fillna(0).astype('float32').clip(lower=0.0, upper=1.0)

    if 'is_manual_renew' in merged.columns:
        merged['is_manual_renew'] = merged['is_manual_renew'].fillna(0).astype('int8')
    elif 'renewal_segment' in merged.columns:
        merged['is_manual_renew'] = merged['renewal_segment'].eq('Pay_Manual').astype('int8')
    elif 'is_auto_renew' in merged.columns:
        merged['is_manual_renew'] = (1 - merged['is_auto_renew'].fillna(0).astype('int8')).clip(lower=0, upper=1)
    else:
        merged['is_manual_renew'] = 0

    if 'deal_hunter_flag' not in merged.columns:
        if 'price_segment' in merged.columns:
            merged['deal_hunter_flag'] = merged['price_segment'].eq('Deal Hunter < 4.5').astype('int8')
        elif 'amt_per_day' in merged.columns:
            merged['deal_hunter_flag'] = ((merged['amt_per_day'].fillna(0) > 0) & (merged['amt_per_day'].fillna(0) < 4.5)).astype('int8')
        else:
            merged['deal_hunter_flag'] = 0
    else:
        merged['deal_hunter_flag'] = merged['deal_hunter_flag'].fillna(0).astype('int8')

    if 'free_trial_flag' not in merged.columns:
        merged['free_trial_flag'] = (merged['price_segment'].eq('Free Trial / Zero Pay') | merged['expected_renewal_amount'].le(0)).astype('int8')
    else:
        merged['free_trial_flag'] = merged['free_trial_flag'].fillna(0).astype('int8')

    if 'high_skip_flag' not in merged.columns:
        merged['high_skip_flag'] = merged['skip_ratio'].ge(0.5).astype('int8')
    else:
        merged['high_skip_flag'] = merged['high_skip_flag'].fillna(0).astype('int8')

    if 'low_discovery_flag' not in merged.columns:
        merged['low_discovery_flag'] = merged['discovery_ratio'].lt(0.2).astype('int8')
    else:
        merged['low_discovery_flag'] = merged['low_discovery_flag'].fillna(0).astype('int8')

    if 'content_fatigue_flag' not in merged.columns:
        merged['content_fatigue_flag'] = (merged['high_skip_flag'].eq(1) & merged['low_discovery_flag'].eq(1)).astype('int8')
    else:
        merged['content_fatigue_flag'] = merged['content_fatigue_flag'].fillna(0).astype('int8')

    merged['eligible_auto'] = merged['is_manual_renew'].eq(1).astype('int8')
    merged['eligible_upsell'] = (merged['deal_hunter_flag'].eq(1) | merged['free_trial_flag'].eq(1)).astype('int8')
    merged['eligible_engagement'] = (merged['high_skip_flag'].eq(1) | merged['low_discovery_flag'].eq(1) | merged['content_fatigue_flag'].eq(1)).astype('int8')
    merged['baseline_revenue_at_risk_30d'] = (merged['baseline_churn_probability'] * merged['expected_renewal_amount']).astype('float32')
    merged['baseline_retained_revenue_30d'] = ((1 - merged['baseline_churn_probability']) * merged['expected_renewal_amount']).astype('float32')
    return merged


def estimate_lever_parameters(df: pd.DataFrame) -> dict[str, float]:
    population_default = _mean_or_default(df['baseline_churn_probability'], default=0.5)

    manual_mean = _mean_or_default(df.loc[df['eligible_auto'] == 1, 'baseline_churn_probability'], default=population_default)
    auto_mean = _mean_or_default(df.loc[df['renewal_segment'].eq('Pay_Auto-Renew'), 'baseline_churn_probability'], default=population_default)
    auto_gap = max(manual_mean - auto_mean, 0.0)
    auto_prob_delta_if_treated = -float(np.clip(auto_gap, 0.0, 0.15))

    fatigue_mean = _mean_or_default(df.loc[df['eligible_engagement'] == 1, 'baseline_churn_probability'], default=population_default)
    healthy_mean = _mean_or_default(df.loc[df['eligible_engagement'] == 0, 'baseline_churn_probability'], default=population_default)
    engagement_gap = max(fatigue_mean - healthy_mean, 0.0)
    engagement_prob_delta_if_treated = -float(np.clip(engagement_gap, 0.0, 0.18))

    positive_amounts = df.loc[df['expected_renewal_amount'] > 0, 'expected_renewal_amount']
    standard_amounts = df.loc[df['price_segment'].eq('Standard 4.5-6.5'), 'expected_renewal_amount']
    premium_amounts = df.loc[df['price_segment'].eq('Premium >= 6.5'), 'expected_renewal_amount']
    standard_target_amount = _first_non_empty_median([standard_amounts, positive_amounts], default=0.0)
    premium_target_amount = _first_non_empty_median([premium_amounts, positive_amounts], default=standard_target_amount)
    premium_target_amount = max(premium_target_amount, standard_target_amount)

    deal_mean = _mean_or_default(df.loc[df['eligible_upsell'] == 1, 'baseline_churn_probability'], default=population_default)
    paid_stable_mean = _mean_or_default(df.loc[df['eligible_upsell'] == 0, 'baseline_churn_probability'], default=population_default)
    upsell_group_gap = float(np.clip(paid_stable_mean - deal_mean, -0.15, 0.12))

    return {
        'auto_prob_delta_if_treated': auto_prob_delta_if_treated,
        'engagement_prob_delta_if_treated': engagement_prob_delta_if_treated,
        'upsell_group_gap': upsell_group_gap,
        'standard_target_amount': float(standard_target_amount),
        'premium_target_amount': float(premium_target_amount),
        'manual_mean_probability': manual_mean,
        'auto_mean_probability': auto_mean,
        'fatigue_mean_probability': fatigue_mean,
        'healthy_mean_probability': healthy_mean,
        'deal_mean_probability': deal_mean,
        'paid_stable_mean_probability': paid_stable_mean,
    }


def _normalize_config(config: dict | None) -> dict[str, float]:
    normalized = deepcopy(DEFAULT_SCENARIO_CONFIG)
    if config:
        normalized.update(config)
    for key in ('manual_to_auto_share', 'upsell_share', 'engagement_share'):
        normalized[key] = _clip_share(normalized.get(key))
    for key in ('manual_to_auto_cost_per_user', 'upsell_cost_per_user', 'engagement_cost_per_user'):
        normalized[key] = float(max(float(normalized.get(key, 0.0)), 0.0))
    return normalized


def _upsell_target_amount(df: pd.DataFrame, params: dict[str, float]) -> pd.Series:
    current_amount = df['expected_renewal_amount'].astype('float32')
    standard_target = float(params['standard_target_amount'])
    premium_target = float(params['premium_target_amount'])
    base_target = np.where(df['free_trial_flag'].eq(1), standard_target, np.maximum(current_amount.to_numpy(dtype='float32'), standard_target))
    high_value_mask = (df['rfm_segment'].eq('High Value') | df.get('rfm_total_score', pd.Series(0, index=df.index)).fillna(0).ge(7))
    base_target = np.where(high_value_mask, np.maximum(base_target, premium_target), base_target)
    return pd.Series(base_target, index=df.index, dtype='float32')


def simulate_prescriptive_scenario(baseline_df: pd.DataFrame, scenario_config: dict | None = None, lever_parameters: dict | None = None) -> pd.DataFrame:
    config = _normalize_config(scenario_config)
    params = lever_parameters or estimate_lever_parameters(baseline_df)
    df = baseline_df.copy()

    df['auto_treatment_weight'] = df['eligible_auto'].astype('float32') * config['manual_to_auto_share']
    df['engagement_treatment_weight'] = df['eligible_engagement'].astype('float32') * config['engagement_share']
    df['upsell_treatment_weight'] = df['eligible_upsell'].astype('float32') * config['upsell_share']

    df['auto_prob_delta_if_treated'] = float(params['auto_prob_delta_if_treated'])
    df['engagement_prob_delta_if_treated'] = float(params['engagement_prob_delta_if_treated'])

    target_amount = _upsell_target_amount(df, params)
    df['upsell_target_amount_if_treated'] = target_amount
    df['upsell_amount_delta_if_treated'] = ((target_amount - df['expected_renewal_amount']).clip(lower=0.0).astype('float32'))

    uplift_ratio = (df['upsell_amount_delta_if_treated'] / np.maximum(target_amount.to_numpy(dtype='float32'), 1.0)).astype('float32')
    price_shock_penalty = np.clip(0.08 * uplift_ratio, 0.0, 0.10)
    df['upsell_prob_delta_if_treated'] = np.clip(float(params['upsell_group_gap']) + price_shock_penalty, -0.15, 0.12).astype('float32')

    df['auto_probability_delta'] = (df['auto_treatment_weight'] * df['auto_prob_delta_if_treated']).astype('float32')
    df['engagement_probability_delta'] = (df['engagement_treatment_weight'] * df['engagement_prob_delta_if_treated']).astype('float32')
    df['upsell_probability_delta'] = (df['upsell_treatment_weight'] * df['upsell_prob_delta_if_treated']).astype('float32')
    df['upsell_amount_delta'] = (df['upsell_treatment_weight'] * df['upsell_amount_delta_if_treated']).astype('float32')

    df['simulated_expected_renewal_amount'] = (df['expected_renewal_amount'] + df['upsell_amount_delta']).astype('float32')
    df['simulated_churn_probability'] = np.clip(
        df['baseline_churn_probability']
        + df['auto_probability_delta']
        + df['engagement_probability_delta']
        + df['upsell_probability_delta'],
        1e-4,
        1 - 1e-4,
    ).astype('float32')
    df['simulated_revenue_at_risk_30d'] = (df['simulated_churn_probability'] * df['simulated_expected_renewal_amount']).astype('float32')
    df['price_only_retained_revenue_30d'] = ((1 - df['baseline_churn_probability']) * df['simulated_expected_renewal_amount']).astype('float32')
    df['simulated_retained_revenue_30d'] = ((1 - df['simulated_churn_probability']) * df['simulated_expected_renewal_amount']).astype('float32')
    df['risk_effect_revenue_delta_30d'] = (df['simulated_retained_revenue_30d'] - df['price_only_retained_revenue_30d']).astype('float32')
    df['upsell_effect_revenue_delta_30d'] = (df['price_only_retained_revenue_30d'] - df['baseline_retained_revenue_30d']).astype('float32')
    df['total_retained_revenue_delta_30d'] = (df['simulated_retained_revenue_30d'] - df['baseline_retained_revenue_30d']).astype('float32')
    df['simulated_risk_band'] = risk_band_from_probability(df['simulated_churn_probability'])

    df['manual_to_auto_cost'] = (df['auto_treatment_weight'] * config['manual_to_auto_cost_per_user']).astype('float32')
    df['upsell_cost'] = (df['upsell_treatment_weight'] * config['upsell_cost_per_user']).astype('float32')
    df['engagement_cost'] = (df['engagement_treatment_weight'] * config['engagement_cost_per_user']).astype('float32')
    df['campaign_cost'] = (df['manual_to_auto_cost'] + df['upsell_cost'] + df['engagement_cost']).astype('float32')
    return df


def summarize_scenario(member_level_df: pd.DataFrame, scenario_config: dict[str, float], lever_parameters: dict[str, float]) -> dict[str, Any]:
    summary = {
        'score_month': int(member_level_df['target_month'].iloc[0]),
        'population_users': int(member_level_df['msno'].nunique()),
        'eligible_auto_users': int(member_level_df['eligible_auto'].sum()),
        'eligible_upsell_users': int(member_level_df['eligible_upsell'].sum()),
        'eligible_engagement_users': int(member_level_df['eligible_engagement'].sum()),
        'impacted_auto_user_equivalent': float(member_level_df['auto_treatment_weight'].sum()),
        'impacted_upsell_user_equivalent': float(member_level_df['upsell_treatment_weight'].sum()),
        'impacted_engagement_user_equivalent': float(member_level_df['engagement_treatment_weight'].sum()),
        'baseline_avg_churn_probability': float(member_level_df['baseline_churn_probability'].mean()),
        'simulated_avg_churn_probability': float(member_level_df['simulated_churn_probability'].mean()),
        'baseline_revenue_at_risk_30d': float(member_level_df['baseline_revenue_at_risk_30d'].sum()),
        'simulated_revenue_at_risk_30d': float(member_level_df['simulated_revenue_at_risk_30d'].sum()),
        'baseline_retained_revenue_30d': float(member_level_df['baseline_retained_revenue_30d'].sum()),
        'simulated_retained_revenue_30d': float(member_level_df['simulated_retained_revenue_30d'].sum()),
        'saved_revenue_from_risk_reduction_30d': float(member_level_df['risk_effect_revenue_delta_30d'].clip(lower=0).sum()),
        'revenue_loss_from_risk_increase_30d': float((-member_level_df['risk_effect_revenue_delta_30d']).clip(lower=0).sum()),
        'incremental_upsell_revenue_30d': float(member_level_df['upsell_effect_revenue_delta_30d'].sum()),
        'campaign_cost_30d': float(member_level_df['campaign_cost'].sum()),
        'scenario_config': deepcopy(scenario_config),
        'lever_parameters': deepcopy(lever_parameters),
    }
    summary['revenue_at_risk_delta_30d'] = summary['simulated_revenue_at_risk_30d'] - summary['baseline_revenue_at_risk_30d']
    summary['retained_revenue_delta_30d'] = summary['simulated_retained_revenue_30d'] - summary['baseline_retained_revenue_30d']
    summary['net_value_after_cost_30d'] = summary['retained_revenue_delta_30d'] - summary['campaign_cost_30d']
    return summary


In [ ]:
def build_simulation_units(member_level_df: pd.DataFrame) -> pd.DataFrame:
    group_cols = [col for col in SIMULATION_UNIT_KEYS if col in member_level_df.columns]
    units = (
        member_level_df.groupby(group_cols, dropna=False, as_index=False)
        .agg(
            users=('msno', 'nunique'),
            baseline_churn_probability=('baseline_churn_probability', 'mean'),
            auto_probability_delta=('auto_probability_delta', 'mean'),
            engagement_probability_delta=('engagement_probability_delta', 'mean'),
            upsell_probability_delta=('upsell_probability_delta', 'mean'),
            expected_renewal_amount=('expected_renewal_amount', 'mean'),
            upsell_amount_delta=('upsell_amount_delta', 'mean'),
            campaign_cost=('campaign_cost', 'sum'),
            baseline_revenue_at_risk_30d=('baseline_revenue_at_risk_30d', 'sum'),
            baseline_retained_revenue_30d=('baseline_retained_revenue_30d', 'sum'),
            simulated_revenue_at_risk_30d=('simulated_revenue_at_risk_30d', 'sum'),
            simulated_retained_revenue_30d=('simulated_retained_revenue_30d', 'sum'),
        )
        .sort_values(['users', 'baseline_revenue_at_risk_30d'], ascending=[False, False])
        .reset_index(drop=True)
    )
    units['users'] = units['users'].astype('int32')
    return units


def _clipped_normal(rng: np.random.Generator, mean: float, std: float, low: float, high: float) -> float:
    return float(np.clip(rng.normal(mean, std), low, high))


def _metric_percentiles(series: pd.Series) -> dict[str, float]:
    return {
        'mean': float(series.mean()),
        'std': float(series.std(ddof=0)),
        'p05': float(series.quantile(0.05)),
        'p25': float(series.quantile(0.25)),
        'p50': float(series.quantile(0.50)),
        'p75': float(series.quantile(0.75)),
        'p95': float(series.quantile(0.95)),
    }


def summarize_monte_carlo(mc_runs_df: pd.DataFrame, deterministic_summary: dict[str, Any], mc_config: dict[str, float], score_month: int) -> tuple[dict[str, Any], pd.DataFrame]:
    metric_map = {
        'baseline_retained_revenue_30d': 'Baseline Revenue',
        'scenario_retained_revenue_30d': 'Scenario Revenue',
        'incremental_upsell_revenue_30d': 'Incremental Upsell Revenue',
        'saved_revenue_from_risk_reduction_30d': 'Saved Revenue',
        'campaign_cost_30d': 'Campaign Cost',
        'net_value_after_cost_30d': 'Net Value After Cost',
        'baseline_churn_prob_pct': 'Baseline Churn %',
        'scenario_churn_prob_pct': 'Scenario Churn %',
    }
    rows = []
    metrics_payload = {}
    for column, label in metric_map.items():
        stats = _metric_percentiles(mc_runs_df[column])
        stats['metric'] = label
        stats['column'] = column
        rows.append(stats)
        metrics_payload[column] = {key: value for key, value in stats.items() if key not in {'metric', 'column'}}
    percentiles_df = pd.DataFrame(rows)[['metric', 'column', 'mean', 'std', 'p05', 'p25', 'p50', 'p75', 'p95']]
    summary_payload = {
        'score_month': score_month,
        'n_iterations': int(mc_config['n_iterations']),
        'seed': int(mc_config['seed']),
        'beta_concentration': float(mc_config['beta_concentration']),
        'deterministic_summary': deterministic_summary,
        'monte_carlo_metrics': metrics_payload,
        'probability_scenario_beats_baseline': float((mc_runs_df['scenario_retained_revenue_30d'] > mc_runs_df['baseline_retained_revenue_30d']).mean()),
        'probability_net_positive': float((mc_runs_df['net_value_after_cost_30d'] > 0).mean()),
    }
    return summary_payload, percentiles_df


def run_monte_carlo(member_level_df: pd.DataFrame, mc_config: dict[str, float]) -> tuple[pd.DataFrame, pd.DataFrame, dict[str, Any], pd.DataFrame]:
    units_df = build_simulation_units(member_level_df)
    if units_df.empty:
        raise ValueError('Không có simulation unit để chạy Monte Carlo.')

    cfg = deepcopy(mc_config)
    rng = np.random.default_rng(int(cfg['seed']))

    users = units_df['users'].to_numpy(dtype='int32')
    total_users = max(int(users.sum()), 1)
    base_prob = units_df['baseline_churn_probability'].to_numpy(dtype='float32')
    auto_delta = units_df['auto_probability_delta'].to_numpy(dtype='float32')
    engage_delta = units_df['engagement_probability_delta'].to_numpy(dtype='float32')
    upsell_delta = units_df['upsell_probability_delta'].to_numpy(dtype='float32')
    base_amount = units_df['expected_renewal_amount'].to_numpy(dtype='float32')
    upsell_amount_delta = units_df['upsell_amount_delta'].to_numpy(dtype='float32')
    campaign_cost_total = float(units_df['campaign_cost'].sum())
    concentration = max(float(cfg['beta_concentration']), 20.0)

    rows = []
    for run_id in range(int(cfg['n_iterations'])):
        auto_mult = _clipped_normal(rng, 1.0, float(cfg['auto_effect_std']), 0.4, 1.6)
        engagement_mult = _clipped_normal(rng, 1.0, float(cfg['engagement_effect_std']), 0.4, 1.8)
        upsell_risk_mult = _clipped_normal(rng, 1.0, float(cfg['upsell_risk_std']), 0.4, 1.8)
        upsell_amount_mult = _clipped_normal(rng, 1.0, float(cfg['upsell_amount_std']), 0.7, 1.4)

        scenario_prob = np.clip(
            base_prob + auto_delta * auto_mult + engage_delta * engagement_mult + upsell_delta * upsell_risk_mult,
            1e-4,
            1 - 1e-4,
        )
        scenario_amount = np.maximum(base_amount + upsell_amount_delta * upsell_amount_mult, 0.0)

        baseline_retention = 1.0 - base_prob
        scenario_retention = 1.0 - scenario_prob

        baseline_retention_draw = rng.beta(
            np.maximum(baseline_retention * concentration, 1.0),
            np.maximum((1.0 - baseline_retention) * concentration, 1.0),
        )
        scenario_retention_draw = rng.beta(
            np.maximum(scenario_retention * concentration, 1.0),
            np.maximum((1.0 - scenario_retention) * concentration, 1.0),
        )

        baseline_renewals = rng.binomial(users, baseline_retention_draw)
        scenario_renewals = rng.binomial(users, scenario_retention_draw)

        baseline_revenue = float((baseline_renewals * base_amount).sum())
        price_only_revenue = float((baseline_renewals * scenario_amount).sum())
        scenario_revenue = float((scenario_renewals * scenario_amount).sum())
        risk_saved = float(scenario_revenue - price_only_revenue)
        upsell_gain = float(price_only_revenue - baseline_revenue)

        rows.append({
            'run_id': run_id,
            'auto_effect_multiplier': auto_mult,
            'engagement_effect_multiplier': engagement_mult,
            'upsell_risk_multiplier': upsell_risk_mult,
            'upsell_amount_multiplier': upsell_amount_mult,
            'baseline_retained_revenue_30d': baseline_revenue,
            'scenario_retained_revenue_30d': scenario_revenue,
            'incremental_upsell_revenue_30d': upsell_gain,
            'saved_revenue_from_risk_reduction_30d': risk_saved,
            'campaign_cost_30d': campaign_cost_total,
            'net_value_after_cost_30d': scenario_revenue - baseline_revenue - campaign_cost_total,
            'baseline_churn_prob_pct': 100.0 * (1.0 - baseline_renewals.sum() / total_users),
            'scenario_churn_prob_pct': 100.0 * (1.0 - scenario_renewals.sum() / total_users),
        })

    mc_runs_df = pd.DataFrame(rows)
    deterministic_summary = {
        'baseline_retained_revenue_30d': float(member_level_df['baseline_retained_revenue_30d'].sum()),
        'scenario_retained_revenue_30d': float(member_level_df['simulated_retained_revenue_30d'].sum()),
        'incremental_upsell_revenue_30d': float(member_level_df['upsell_effect_revenue_delta_30d'].sum()),
        'saved_revenue_from_risk_reduction_30d': float(member_level_df['risk_effect_revenue_delta_30d'].sum()),
        'campaign_cost_30d': campaign_cost_total,
        'net_value_after_cost_30d': float(member_level_df['simulated_retained_revenue_30d'].sum() - member_level_df['baseline_retained_revenue_30d'].sum() - campaign_cost_total),
        'baseline_churn_prob_pct': float(member_level_df['baseline_churn_probability'].mean() * 100.0),
        'scenario_churn_prob_pct': float(member_level_df['simulated_churn_probability'].mean() * 100.0),
    }
    summary_payload, percentiles_df = summarize_monte_carlo(mc_runs_df, deterministic_summary, cfg, int(member_level_df['target_month'].iloc[0]))
    return units_df, mc_runs_df, summary_payload, percentiles_df


def run_tab3_monte_carlo_artifacts(feature_store_root_hint: str | Path | None = None, tab2_root_hint: str | Path | None = None, output_dir: str | Path | None = None, score_month: int = SCORE_MONTH, scenario_config: dict[str, float] | None = None, mc_config: dict[str, float] | None = None) -> dict[str, Any]:
    feature_store_dir = resolve_feature_store_dir(feature_store_root_hint, score_month=score_month)
    tab2_dir = resolve_tab2_artifacts_dir(tab2_root_hint, score_month=score_month)
    output_dir = ensure_output_dir(output_dir or OUTPUT_DIR)

    feature_snapshot = load_feature_snapshot(feature_store_dir, score_month=score_month)
    scored_df = pd.read_parquet(tab2_dir / f'tab2_test_scored_{score_month}.parquet')
    baseline_df = _prepare_baseline_dataframe(feature_snapshot, scored_df)
    lever_parameters = estimate_lever_parameters(baseline_df)

    normalized_config = _normalize_config(scenario_config)
    normalized_mc_config = deepcopy(MC_CONFIG)
    if mc_config:
        normalized_mc_config.update(mc_config)

    member_level_df = simulate_prescriptive_scenario(
        baseline_df,
        scenario_config=normalized_config,
        lever_parameters=lever_parameters,
    )
    deterministic_summary = summarize_scenario(member_level_df, normalized_config, lever_parameters)
    units_df, mc_runs_df, mc_summary_payload, percentiles_df = run_monte_carlo(member_level_df, normalized_mc_config)

    output_paths = {
        'member_level': output_dir / f'tab3_mc_member_level_{score_month}.parquet',
        'simulation_units': output_dir / f'tab3_mc_units_{score_month}.parquet',
        'monte_carlo_runs': output_dir / f'tab3_monte_carlo_runs_{score_month}.parquet',
        'monte_carlo_percentiles': output_dir / f'tab3_monte_carlo_percentiles_{score_month}.parquet',
        'monte_carlo_summary': output_dir / f'tab3_monte_carlo_summary_{score_month}.json',
        'deterministic_summary': output_dir / f'tab3_deterministic_summary_{score_month}.json',
    }

    member_level_df.to_parquet(output_paths['member_level'], index=False)
    units_df.to_parquet(output_paths['simulation_units'], index=False)
    mc_runs_df.to_parquet(output_paths['monte_carlo_runs'], index=False)
    percentiles_df.to_parquet(output_paths['monte_carlo_percentiles'], index=False)
    write_json(output_paths['monte_carlo_summary'], mc_summary_payload)
    write_json(output_paths['deterministic_summary'], deterministic_summary)

    manifest_path = write_manifest(
        output_dir=output_dir,
        notebook_name='kkbox-simulation-monte-carlo.ipynb',
        artifact_type='tab3_monte_carlo',
        input_paths={
            'feature_store_dir': feature_store_dir,
            'feature_snapshot': feature_store_dir / f'test_features_bi_{score_month}_full.parquet',
            'tab2_artifacts_dir': tab2_dir,
            'tab2_test_scored': tab2_dir / f'tab2_test_scored_{score_month}.parquet',
        },
        output_paths=output_paths,
        metadata={
            'score_month': score_month,
            'scenario_config': normalized_config,
            'mc_config': normalized_mc_config,
            'lever_parameters': lever_parameters,
            'population_users': int(member_level_df['msno'].nunique()),
            'simulation_unit_count': int(len(units_df)),
        },
    )
    output_paths['manifest'] = manifest_path

    return {
        'feature_store_dir': feature_store_dir,
        'tab2_artifacts_dir': tab2_dir,
        'output_dir': output_dir,
        'baseline_df': baseline_df,
        'member_level_df': member_level_df,
        'deterministic_summary': deterministic_summary,
        'simulation_units': units_df,
        'monte_carlo_runs': mc_runs_df,
        'monte_carlo_summary': mc_summary_payload,
        'percentiles_df': percentiles_df,
        'output_paths': output_paths,
    }


In [ ]:
def slugify_scenario_id(value: str) -> str:
    text = ''.join(ch.lower() if ch.isalnum() else '-' for ch in str(value).strip())
    while '--' in text:
        text = text.replace('--', '-')
    return text.strip('-') or 'scenario'


def run_tab3_monte_carlo_catalog(
    scenario_cases: list[dict],
    feature_store_root_hint: str | Path | None = None,
    tab2_root_hint: str | Path | None = None,
    output_dir: str | Path | None = None,
    score_month: int = SCORE_MONTH,
    default_scenario_id: str | None = None,
    mc_config: dict[str, float] | None = None,
) -> dict:
    if not scenario_cases:
        raise ValueError('scenario_cases phai co it nhat 1 preset')

    output_dir = ensure_output_dir(output_dir or OUTPUT_DIR)
    default_id = slugify_scenario_id(default_scenario_id or scenario_cases[0].get('scenario_id') or 'default')
    catalog_rows: list[dict] = []
    results: dict[str, dict] = {}

    for index, case in enumerate(scenario_cases):
        case_id = slugify_scenario_id(case.get('scenario_id') or f'scenario-{index + 1}')
        scenario_config = {
            'manual_to_auto_share': 0.30,
            'upsell_share': 0.20,
            'engagement_share': 0.25,
            'manual_to_auto_cost_per_user': 0.0,
            'upsell_cost_per_user': 0.0,
            'engagement_cost_per_user': 0.0,
            **deepcopy(case.get('scenario_config', {})),
        }
        case_output_dir = output_dir if case_id == default_id else output_dir / 'scenarios' / case_id
        result = run_tab3_monte_carlo_artifacts(
            feature_store_root_hint=feature_store_root_hint,
            tab2_root_hint=tab2_root_hint,
            output_dir=case_output_dir,
            score_month=score_month,
            scenario_config=scenario_config,
            mc_config=mc_config,
        )
        results[case_id] = result
        catalog_rows.append(
            {
                'scenario_id': case_id,
                'label': case.get('label') or case_id.replace('-', ' ').title(),
                'description': case.get('description') or '',
                'monte_carlo_subdir': '.' if case_id == default_id else f'scenarios/{case_id}',
                'scenario_config': deepcopy(scenario_config),
                'scenario_inputs': {
                    'auto_shift_pct': float(scenario_config['manual_to_auto_share'] * 100.0),
                    'upsell_shift_pct': float(scenario_config['upsell_share'] * 100.0),
                    'skip_shift_pct': float(scenario_config['engagement_share'] * 100.0),
                },
            }
        )

    catalog_payload = {
        'default_scenario_id': default_id,
        'score_month': score_month,
        'scenarios': catalog_rows,
    }
    catalog_path = write_json(output_dir / 'scenario_catalog.json', catalog_payload)
    return {
        'output_dir': output_dir,
        'catalog_path': catalog_path,
        'catalog': catalog_payload,
        'results': results,
    }


In [ ]:
DEFAULT_SCENARIO_ID = 'balanced-demo'
SCENARIO_CASES = [
    {
        'scenario_id': 'balanced-demo',
        'label': 'Balanced demo',
        'description': 'Preset mac dinh cho web demo.',
        'scenario_config': {
            'manual_to_auto_share': 0.20,
            'upsell_share': 0.15,
            'engagement_share': 0.25,
        },
    },
    {
        'scenario_id': 'auto-renew-push',
        'label': 'Auto-renew push',
        'description': 'Tang conversion manual sang auto-renew cho nhom co nguy co roi bo.',
        'scenario_config': {
            'manual_to_auto_share': 0.35,
            'upsell_share': 0.10,
            'engagement_share': 0.20,
        },
    },
    {
        'scenario_id': 'upsell-growth',
        'label': 'Upsell growth',
        'description': 'Tang upsell cho nhom deal/free-trial co kha nang nang gia tri gia han.',
        'scenario_config': {
            'manual_to_auto_share': 0.15,
            'upsell_share': 0.35,
            'engagement_share': 0.15,
        },
    },
    {
        'scenario_id': 'engagement-rescue',
        'label': 'Engagement rescue',
        'description': 'Tap trung giam high-skip va cai thien discovery cho nhom content fatigue.',
        'scenario_config': {
            'manual_to_auto_share': 0.10,
            'upsell_share': 0.10,
            'engagement_share': 0.40,
        },
    },
]

catalog_result = run_tab3_monte_carlo_catalog(
    scenario_cases=SCENARIO_CASES,
    feature_store_root_hint=FEATURE_STORE_ROOT_HINT,
    tab2_root_hint=TAB2_ARTIFACTS_ROOT_HINT,
    output_dir=OUTPUT_DIR,
    score_month=SCORE_MONTH,
    default_scenario_id=DEFAULT_SCENARIO_ID,
    mc_config=MC_CONFIG,
)

default_result = catalog_result['results'][DEFAULT_SCENARIO_ID]
display(pd.DataFrame(catalog_result['catalog']['scenarios']))
display(pd.DataFrame([default_result['deterministic_summary']]))
display(default_result['percentiles_df'])
display(default_result['simulation_units'].head(10))
display(default_result['monte_carlo_runs'].head())
print('Output dir:', catalog_result['output_dir'])
print('Scenario catalog:', catalog_result['catalog_path'])
print('Saved files for default scenario:')
for key, value in default_result['output_paths'].items():
    print(f'  - {key}: {value}')
